# Sistema multiagente de soporte interno — TalentoHub

**Proyecto Integrador M3**

Un orquestador clasifica la intencion de cada consulta mediante *function calling* y
la enruta condicionalmente a uno de tres agentes RAG especializados. Todo el flujo se
implementa con LangChain y queda trazado en Langfuse.

## Contexto

TalentoHub es una empresa SaaS de gestion de nomina y personal para LATAM. Su mesa de
soporte interna recibe consultas de empleados que se enrutan mal: preguntas de nomina
llegan a Tecnologia, problemas de acceso llegan a Recursos Humanos. Este sistema
clasifica y enruta automaticamente.

## Arquitectura

```
Consulta del empleado
        |
        v
  ORQUESTADOR  (create_agent + function calling, temperature=0)
        |
        +-- consultar_recursos_humanos --> Agente RAG HR      (60 chunks)
        +-- consultar_tecnologia        --> Agente RAG IT      (60 chunks)
        +-- consultar_finanzas          --> Agente RAG Finanzas (60 chunks)
        +-- ninguna tool                --> pide aclaracion
        |
        v
   Respuesta fundamentada
```

Cada agente RAG es una cadena LCEL: `retriever -> prompt -> modelo -> parser`.

## Nota sobre la organizacion del codigo

La implementacion vive en `src/multi_agent_system.py`. Este notebook la importa en
lugar de duplicarla, para mantener una sola fuente de verdad entre el notebook y los
scripts de linea de comandos. Los entregables admiten ambas formas; esta evita que las
dos versiones se desincronicen.

## Orden de ejecucion

Antes de correr este notebook hay que construir los indices una sola vez:

```
python scripts/build_index.py hr
python scripts/build_index.py tech
python scripts/build_index.py finance
```

Luego se ejecutan las celdas de arriba hacia abajo.

---
# 1. Setup e imports

Se cargan las credenciales desde `.env` (nunca hardcodeadas) y se importa el sistema.

**Decision tecnica — modelos.** `gpt-4o-mini` para orquestador y agentes:
la clasificacion entre tres dominios bien delimitados no requiere un modelo mayor, y el
costo por consulta baja un orden de magnitud. `text-embedding-3-small` para embeddings,
suficiente para un corpus de 180 chunks en espanol.

**Decision tecnica — temperature=0.** Tanto en el orquestador como en los agentes. En el
orquestador porque la clasificacion debe ser reproducible: la misma consulta debe
enrutarse siempre igual. En los agentes porque la respuesta debe ceñirse al contexto
recuperado, no elaborar sobre el.

In [21]:
import json
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))

from src.multi_agent_system import (
    CADENAS,
    CHAT_MODEL,
    DOMINIOS,
    EMBEDDING_MODEL,
    HERRAMIENTAS,
    K,
    PROMPT_ORQUESTADOR,
    PROMPT_RAG,
    atender_consulta,
    cargar_retriever,
    langfuse,
    orquestador,
)

print(f"Modelo de chat      : {CHAT_MODEL}")
print(f"Modelo de embeddings: {EMBEDDING_MODEL}")
print(f"Chunks recuperados  : k={K}")
print(f"Dominios            : {list(DOMINIOS)}")
print(f"Conexion Langfuse   : {langfuse.auth_check()}")

Modelo de chat      : gpt-4o-mini
Modelo de embeddings: text-embedding-3-small
Chunks recuperados  : k=3
Dominios            : ['hr', 'tech', 'finance']
Conexion Langfuse   : True


---
# 2. Carga de documentos y vector stores

## Estrategia de chunking

**Chunking estructural por bloque Q&A.** Cada chunk es una pregunta con su respuesta,
delimitados por linea en blanco. Se eligio sobre el chunking por ventana de caracteres
por tres razones:

1. **Integridad semantica.** Un chunk nunca parte una politica a la mitad. Con ventana
   fija, la respuesta a "cuantos dias de vacaciones" podria quedar cortada entre dos
   chunks y ninguno responderia completo.
2. **Conteo determinista.** Bloques = chunks. El conteo no depende de donde caiga un
   contador de caracteres, lo que hace verificable el minimo de 50 por dominio.
3. **Afinidad con la consulta.** Las consultas reales de empleados se parecen mas a
   preguntas que a prosa de manual. Tener la pregunta dentro del chunk mejora la
   similitud coseno frente a un fragmento de texto corrido.

Cada bloque abre con una etiqueta de seccion (`[Vacaciones]`, `[VPN]`) que aporta senal
tematica al embedding y queda como metadata para depurar recuperaciones.

**Salvaguarda.** Un `RecursiveCharacterTextSplitter` subdivide cualquier bloque que
supere 500 tokens. Con este corpus no se activa (el bloque mayor tiene 55 palabras),
pero deja el pipeline a prueba de documentos futuros con parrafos largos.

## Decision tecnica — vector store

Se uso `InMemoryVectorStore` de `langchain-core` en lugar de FAISS. Razones:

- **`langchain-community` fue discontinuado en mayo de 2026** y no existe un paquete
  sucesor oficial de FAISS para Python. Usarlo introduciria una dependencia sin
  mantenimiento en el entregable.
- **`faiss-cpu` compila codigo nativo.** Es la dependencia mas fragil del stack: su
  disponibilidad depende de que exista wheel para la version de Python y la plataforma
  de quien ejecute el repo. Los entregables exigen un repo autocontenido y ejecutable.
- **A esta escala no hay diferencia de rendimiento.** La ventaja de FAISS es la busqueda
  aproximada, relevante por encima de decenas de miles de vectores. Con 180 chunks la
  busqueda exacta por coseno es instantanea.

La interfaz `Retriever` de LangChain es identica en ambos casos, asi que cambiar de
vector store seria una linea de configuracion. Esa intercambiabilidad es precisamente el
argumento a favor de usar un framework de orquestacion.

El indice se persiste en disco (`indexes/*.json`) para no repagar los embeddings en cada
ejecucion.

In [22]:
for dominio, etiqueta in DOMINIOS.items():
    retriever = cargar_retriever(dominio)
    total = len(retriever.vectorstore.store)
    print(f"{etiqueta:<20} {total} chunks indexados")

Recursos Humanos     60 chunks indexados
Tecnologia           60 chunks indexados
Finanzas             60 chunks indexados


In [23]:
# Inspeccion de un chunk: contenido y metadata
muestra = cargar_retriever("hr").invoke("politica de vacaciones")[0]

print("CONTENIDO:")
print(muestra.page_content)
print("\nMETADATA:")
print(muestra.metadata)

CONTENIDO:
[Vacaciones] ¿Cuántos días de vacaciones me corresponden al año?
Los empleados con contrato a término indefinido acumulan 15 días hábiles de vacaciones por cada año completo trabajado, según la política HR-201. La acumulación es proporcional: 1,25 días por mes trabajado. Los contratos a término fijo de más de seis meses acumulan bajo el mismo esquema.

METADATA:
{'dominio': 'hr', 'fuente': 'politicas_vacaciones_ausencias.md', 'seccion': 'Vacaciones'}


---
# 3. Definicion de los agentes RAG especializados

Cada agente es una cadena LCEL con cuatro etapas:

```
retriever -> prompt -> modelo -> parser de texto
```

## Decision tecnica — k=3

Se recuperan tres chunks por consulta. Las pruebas de recuperacion aisladas mostraron
que en varias consultas el chunk que responde exactamente no siempre queda en primera
posicion: las brechas de similitud entre el primero y el segundo llegan a ser de 0.016.

Ejemplo real medido sobre el corpus de HR con la consulta *"cuantos dias de vacaciones
tengo"*:

| Posicion | Chunk | Similitud |
|---|---|---|
| 1 | Las vacaciones se cuentan en dias habiles o calendario | 0.6511 |
| 2 | **Cuantos dias de vacaciones me corresponden al ano** | 0.6348 |
| 3 | Puedo acumular vacaciones de un ano para otro | 0.5982 |

Con `k=1` esa consulta habria respondido mal. Con `k=3` el chunk correcto entra al
contexto y el modelo responde bien. Subir mas alla de 3 aumentaria tokens y ruido sin
beneficio observado en un corpus de bloques cortos y autocontenidos.

## Decision tecnica — prompt restrictivo

El prompt instruye responder **unicamente** con el contexto recuperado y declarar
explicitamente cuando el contexto no alcanza. Es la defensa principal contra
alucinaciones: sin esa restriccion, el modelo completa con conocimiento general sobre
politicas laborales que no corresponden a TalentoHub.

In [32]:
print(PROMPT_RAG.messages[0].prompt.template)

Eres un asistente de soporte interno de TalentoHub, especializado en {dominio}.

Responde la pregunta del empleado usando UNICAMENTE la informacion del contexto.
Si el contexto no contiene la respuesta, dilo explicitamente y sugiere el canal
adecuado. No inventes politicas, plazos ni montos.
Responde en espanol, de forma breve y concreta.

Contexto:
{context}

Pregunta: {question}

Respuesta:


In [25]:
# Cada agente responde aislado, sin pasar por el orquestador
respuesta = CADENAS["tech"].invoke("que hago si pierdo el celular con el autenticador")
print(respuesta)

Abre un ticket de prioridad alta en la mesa de servicio. TI revoca el dispositivo anterior y te habilita un registro nuevo previa verificación de identidad. Guarda los códigos de respaldo en un lugar seguro para evitar esta situación.


---
# 4. Orquestador y enrutamiento condicional

## Decision tecnica — function calling sobre clasificacion por prompt

El orquestador se construye con `create_agent`, que expone las tres cadenas RAG como
*tools* y deja que el modelo emita una llamada a funcion. La alternativa habria sido
pedirle al modelo una etiqueta de texto (`"hr"`, `"tech"`, `"finance"`) y enrutar con un
`if/else`.

Se eligio function calling porque:

- **El contrato lo impone el modelo, no el parseo.** No hay que validar ni normalizar una
  cadena de texto que el modelo podria devolver como `"HR"`, `"Recursos Humanos"` o
  `"hr."`.
- **Extender es agregar una tool.** Sumar un cuarto dominio (Legal) no requiere tocar
  logica de enrutamiento: se anade la tool con su descripcion y el orquestador la
  considera.
- **La decision queda registrada estructuralmente**, como un `tool_call` en el historial
  de mensajes, y de ahi se extrae el intent para la traza.

## Las descripciones de las tools son el mecanismo de clasificacion

El modelo no ve el corpus: decide solo con la descripcion de cada tool. Por eso enumeran
temas concretos en vez de describir el area en abstracto. La frontera entre dominios se
resuelve ahi: "nomina y su calendario de pago" esta en HR, mientras que un fallo de
acceso al portal de nomina cae en Tecnologia.

## Manejo de casos fuera de alcance

El system prompt instruye no llamar ninguna tool cuando la consulta no corresponde a
ningun dominio, y pedir aclaracion. Ese camino produce `intent = "fuera_de_alcance"`.

In [26]:
print(PROMPT_ORQUESTADOR)

Eres el orquestador del sistema de soporte interno de TalentoHub.

Tu unica funcion es clasificar la consulta del empleado y delegarla a la
herramienta del area correspondiente. No respondas con conocimiento propio.

- Consultas sobre empleo, vacaciones, licencias, nomina, beneficios, desempeno,
  contratacion o retiro: usa consultar_recursos_humanos.
- Consultas sobre equipos, accesos, contrasenas, VPN, software, redes,
  seguridad informatica o desarrollo: usa consultar_tecnologia.
- Consultas sobre reembolsos, viaticos, compras, facturas de proveedores,
  presupuesto, contratos o pagos: usa consultar_finanzas.

Si la consulta mezcla dos areas, elige la que resuelva la necesidad principal.
Si no corresponde a ninguna area, no llames ninguna herramienta: pide al
empleado que aclare su solicitud.

Devuelve la respuesta de la herramienta al empleado sin agregar informacion.


In [27]:
for herramienta in HERRAMIENTAS:
    print(f"--- {herramienta.name} ---")
    print(herramienta.description)
    print()

--- consultar_recursos_humanos ---
Responde consultas de Recursos Humanos de TalentoHub: vacaciones, licencias,
incapacidades, nomina y su calendario de pago, beneficios, medicina prepagada,
evaluacion de desempeno, plan de carrera, onboarding, certificados laborales,
renuncia y liquidacion.

--- consultar_tecnologia ---
Responde consultas de Tecnologia de TalentoHub: contrasenas, accesos y
permisos, autenticacion de dos factores, VPN, correo, equipos de computo y su
soporte, instalacion de software, redes y wifi, seguridad de la informacion,
repositorios de codigo, ambientes e infraestructura.

--- consultar_finanzas ---
Responde consultas de Finanzas de TalentoHub: reembolso de gastos, viaticos
y anticipos de viaje, tarjeta corporativa, ordenes de compra y proveedores,
facturacion y cuentas por pagar, presupuesto y centros de costo, cierre
contable, contratos y controles financieros.



---
# 5. Pruebas y ejemplos

## 5.1 Una consulta por dominio, mas el caso fuera de alcance

In [28]:
ejemplos = [
    "cuantos dias de vacaciones me dan al ano",
    "la vpn me tira error 502",
    "cuanto se demora el pago de un reembolso",
    "cual es la capital de Australia",
]

for consulta in ejemplos:
    resultado = atender_consulta(consulta)
    print(f"CONSULTA : {resultado['query']}")
    print(f"INTENT   : {resultado['intent']}")
    print(f"RESPUESTA: {resultado['response']}")
    print("-" * 78)

langfuse.flush()

CONSULTA : cuantos dias de vacaciones me dan al ano
INTENT   : hr
RESPUESTA: Te corresponden 15 días hábiles de vacaciones por cada año completo trabajado, si tienes un contrato a término indefinido. Si tu contrato es a término fijo de más de seis meses, acumulas bajo el mismo esquema.
------------------------------------------------------------------------------
CONSULTA : la vpn me tira error 502
INTENT   : tech
RESPUESTA: Un error 502 indica que la puerta de enlace no está respondiendo, normalmente por mantenimiento o saturación del concentrador. Te recomiendo revisar la página de estado de servicios. Si el problema persiste por más de 15 minutos, por favor abre un ticket para reportarlo.
------------------------------------------------------------------------------
CONSULTA : cuanto se demora el pago de un reembolso
INTENT   : finance
RESPUESTA: Los reembolsos aprobados antes del día 20 se pagan en el ciclo de la primera quincena del mes siguiente. Los radicados después de esa fech

## 5.2 Golden dataset

`test_queries.json` contiene 14 consultas con su intencion esperada. Cubre los tres
dominios con formulaciones directas y parafraseadas, y cinco casos borde:

| Tipo de caso borde | Consulta | Por que es dificil |
|---|---|---|
| Ambiguo HR/Finanzas | "me pagaron mal la quincena" | Suena a Finanzas; la politica esta en HR |
| Ambiguo IT/HR | "no puedo entrar al portal de nomina" | Menciona nomina; el problema es de acceso |
| Multi-intencion | "voy a renunciar, que hago con el computador" | Toca retiro (HR) y devolucion de equipo (IT) |
| Fuera de alcance | "cual es la capital de Australia" | No debe llamar ninguna tool |
| Mal formada | "hola" | Sin intencion identificable |

El dataset se ejecuta desde linea de comandos con una etiqueta de corrida, de modo que
las 14 trazas queden agrupadas en Langfuse:

```
python scripts/run_golden.py golden-run-v1
```

In [29]:
casos = json.loads(Path("test_queries.json").read_text(encoding="utf-8"))

print(f"{'INTENT ESPERADO':<20}{'TIPO':<26}CONSULTA")
print("-" * 100)
for caso in casos:
    print(f"{caso['expected_intent']:<20}{caso['tipo']:<26}{caso['query']}")
print(f"\nTotal: {len(casos)} casos")

INTENT ESPERADO     TIPO                      CONSULTA
----------------------------------------------------------------------------------------------------
hr                  directa                   cuantos dias de vacaciones me corresponden al ano
hr                  parafraseada              mi esposa esta embarazada, que licencia de paternidad me dan
hr                  directa                   necesito un certificado laboral para el banco
tech                directa                   la vpn me tira error 502 cuando intento conectarme
tech                directa                   olvide mi contrasena corporativa y no puedo entrar
tech                parafraseada              quiero instalar una herramienta que no viene en el computador
finance             directa                   cuanto se demora el pago de un reembolso de gastos
finance             directa                   un proveedor me pregunta por que no le han pagado su factura
finance             parafraseada           

### Resultado de la corrida `golden-run-v1`

**Routing accuracy: 14/14 = 100%**, incluidos los tres casos ambiguos y de
multi-intencion.

**Limitacion conocida.** El corpus, las descripciones de las tools y el golden dataset
fueron construidos por el mismo proceso, lo que introduce sesgo: las consultas de prueba
usan implicitamente el mismo vocabulario que el corpus. Un 100% en estas condiciones
indica que el sistema es coherente consigo mismo, no que sea robusto ante consultas
reales de empleados. Una validacion honesta requeriria consultas recolectadas de la mesa
de soporte real.

In [30]:
resultados = json.loads(
    (Path("outputs") / "golden-run-v1.json").read_text(encoding="utf-8")
)

aciertos = sum(1 for r in resultados if r["acierto"])
print(f"Routing accuracy: {aciertos}/{len(resultados)} = {aciertos / len(resultados):.1%}\n")

print(f"{'OK':<5}{'ESPERADO':<20}{'OBTENIDO':<20}CONSULTA")
print("-" * 100)
for r in resultados:
    print(
        f"{'OK' if r['acierto'] else 'XX':<5}"
        f"{r['expected_intent']:<20}{r['intent']:<20}{r['query'][:45]}"
    )

Routing accuracy: 14/14 = 100.0%

OK   ESPERADO            OBTENIDO            CONSULTA
----------------------------------------------------------------------------------------------------
OK   hr                  hr                  cuantos dias de vacaciones me corresponden al
OK   hr                  hr                  mi esposa esta embarazada, que licencia de pa
OK   hr                  hr                  necesito un certificado laboral para el banco
OK   tech                tech                la vpn me tira error 502 cuando intento conec
OK   tech                tech                olvide mi contrasena corporativa y no puedo e
OK   tech                tech                quiero instalar una herramienta que no viene 
OK   finance             finance             cuanto se demora el pago de un reembolso de g
OK   finance             finance             un proveedor me pregunta por que no le han pa
OK   finance             finance             necesito plata por adelantado para un 

---
# 6. Integracion con Langfuse

## Modelo de trazado

Cada consulta produce exactamente un trace con esta jerarquia:

```
support-request                      <- trace: un request completo del empleado
  |
  +-- orchestrator-routing           <- span: la decision de enrutamiento
        |                               input:  {"query": ...}
        |                               output: {"intent": "hr"}
        |
        +-- generation               <- llamada al LLM que decide (capturada por el handler)
        |
        +-- hr-agent                 <- span: el agente especialista seleccionado
              |                         input:  {"query": ...}
              |                         output: {"response": ...}
              |
              +-- generation         <- llamada al LLM que redacta la respuesta
```

**Decision tecnica — instrumentacion mixta.** Los spans de orquestador y agentes se
abren manualmente con `start_as_current_observation`, para controlar sus nombres y el
contenido de input/output. Las generations las captura automaticamente el
`CallbackHandler` de Langfuse. Instrumentar todo a mano habria significado envolver cada
llamada interna de LangChain; dejarlo todo al handler habria producido spans con nombres
genericos de LangGraph, dificiles de leer en el dashboard.

El nombre del span del agente se arma dinamicamente con el dominio (`hr-agent`,
`tech-agent`, `finance-agent`), asi que en la lista de trazas se ve de un vistazo a que
agente se enruto cada consulta.

## Metadata para depurar enrutamientos incorrectos

`propagate_attributes` propaga `user_id`, `tags` y `expected_intent` a todos los spans
del contexto. Al cerrar, el span raiz registra tambien `actual_intent`. Tener ambos
valores en la misma traza convierte cada error de clasificacion en un caso
autoexplicativo: se abre la traza y se ve que se esperaba y que ocurrio.

Los `tags` incluyen la etiqueta de corrida (`golden-run-v1`), lo que permite filtrar las
14 trazas de una ejecucion completa y compararlas contra corridas posteriores.

## Como depurar con las trazas

1. Filtrar por tag en la lista de trazas.
2. Abrir el trace y revisar el `input` original.
3. Mirar el `output` del span `orchestrator-routing`: si el intent no corresponde, el
   problema esta en las descripciones de las tools o en el system prompt.
4. Si el intent es correcto pero la respuesta es mala, abrir el span del agente y revisar
   que contexto recibio la generation. Un contexto vacio o generico apunta al retriever o
   al corpus del dominio.

In [31]:
resultado = atender_consulta(
    consulta="necesito el certificado de ingresos y retenciones",
    user_id="empleado_demo",
    expected_intent="hr",
    run_tag="demo-notebook",
)
langfuse.flush()

print(f"Intent   : {resultado['intent']}")
print(f"Trace ID : {resultado['trace_id']}")
print(f"\nRespuesta:\n{resultado['response']}")
print(f"\nAbre esta traza en Langfuse filtrando por el tag 'demo-notebook'.")

Intent   : hr
Trace ID : 78b4d81eac5f16de3b2cc9cc4de05f94

Respuesta:
El certificado de ingresos y retenciones del año anterior se publica en Mis Documentos durante el mes de marzo. Para años anteriores, se solicita a nómina por el formulario de Novedades.

Abre esta traza en Langfuse filtrando por el tag 'demo-notebook'.


---
# Resumen de decisiones tecnicas

| Decision | Alternativa descartada | Razon |
|---|---|---|
| Chunking estructural por bloque Q&A | Ventana de caracteres con solapamiento | Integridad semantica, conteo determinista, afinidad con consultas |
| `InMemoryVectorStore` | FAISS via `langchain-community` | Paquete discontinuado y dependencia nativa fragil; sin ventaja a 180 chunks |
| `k=3` | `k=1` | Brechas de similitud de 0.016 medidas; con k=1 una consulta de prueba fallaba |
| Function calling | Clasificacion por etiqueta de texto + if/else | Contrato impuesto por el modelo, extensible sin tocar logica de ruteo |
| `temperature=0` | Valor por defecto | Clasificacion reproducible y respuestas cenidas al contexto |
| Instrumentacion mixta | Todo manual / todo automatico | Spans legibles con nombres controlados, sin envolver internals de LangChain |
| Notebook que importa `src/` | Codigo duplicado en el notebook | Una sola fuente de verdad entre notebook y scripts |

# Limitaciones conocidas

- **El golden dataset tiene sesgo de origen.** Corpus, descripciones de tools y consultas
  de prueba salieron del mismo proceso de construccion. El 100% de routing accuracy mide
  coherencia interna, no robustez frente a consultas reales.
- **Sin memoria conversacional.** Cada consulta se atiende de forma independiente. Un
  seguimiento como "y cuanto se demora eso?" no tiene contexto previo.
- **Consultas multi-intencion se resuelven parcialmente.** El orquestador elige un solo
  dominio. Una consulta que legitimamente requiere dos agentes recibe respuesta de uno.
- **El corpus es sintetico.** Refleja politicas verosimiles de una empresa ficticia, no
  documentacion real.
- **Sin evaluacion automatica de calidad de respuesta.** Se mide el enrutamiento, no si
  la respuesta generada es correcta y completa.